# Evidence-Aware Agentic RAG for Cyber Threat Intelligence — v2
Corpus: MITRE ATT&CK (current **and** v8.2) · CAPEC · CWE · NVD · CISA KEV · NIST SP 800-53 Rev. 5 · CTID mappings (Rev. 5 and Rev. 4)
External questions: CTIBench CTI-ATE and CTI-RCM.

**Two tracks, always reported separately**
* **Text track** – behaviour → technique, report → techniques (CTIBench), CVE description → CWE (CTIBench), attribution, behaviour/name-only claims. A graph lookup cannot answer these.
* **Evidence track** – mitigations/controls, ID claims, and *real* outdated/conflicting evidence (revoked techniques, withdrawn controls, Rev. 4 vs Rev. 5, ATT&CK v8.2 vs current, NVD vs KEV). The no-LLM **graph oracle** is reported here as the reference.

Every stage is resumable and cached on Google Drive.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/cti-evidence-rag'       # unzip the project here
WORK    = '/content/drive/MyDrive/cti-evidence-rag-work'  # data, caches, runs, reports
import os, sys, json
os.environ['CTIRAG_WORK'] = WORK
sys.path.insert(0, PROJECT)
!pip -q install rank_bm25 groq sentence-transformers tabulate

In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
try:
    os.environ['NVD_API_KEY'] = userdata.get('NVD_API_KEY')   # optional, ~10x faster NVD download
except Exception:
    print('No NVD_API_KEY - NVD enrichment is slower (about 6 s per CVE) but works')

In [ ]:
from ctirag.config import Config
def make_cfg(**kw):
    base = dict(work_dir=WORK, nvd_enrich=True, nvd_max_cves=1000, dense_model='BAAI/bge-small-en-v1.5',
                llm_provider='groq', requests_per_minute=25)
    base.update(kw)
    return Config(**base).ensure()
cfg = make_cfg()
cfg.to_dict()

## 2. Download sources, build corpus and benchmark

In [ ]:
from ctirag.runner import prepare
corpus, questions = prepare(cfg)

**Check before going further:** the corpus log must show `MITRE CWE/active` with roughly 1,000+ documents (not ~336 stubs).
If it says *CWE XML missing → stubs*, cwe.mitre.org blocked the download: download `cwec_latest.xml.zip` from https://cwe.mitre.org/data/downloads.html in your browser,
upload it to `WORK/raw/cwe.zip`, and re-run the cell above. The CTIBench RCM questions need the full CWE text.

## 3. Index and LLM

In [ ]:
from ctirag.runner import load_corpus, stratified_sample, run_system, run_tag
from ctirag.index import HybridIndex
from ctirag.benchmark import load_questions
from ctirag.llm import make_llm
from ctirag.evaluate import load_results, score_run
from ctirag.report import make_report, multi_model_table
corpus = load_corpus(cfg)
index = HybridIndex(corpus, cfg.dense_model, cfg.proc_dir)
qs_all = load_questions(cfg)
import collections
print(index.dense_kind, len(corpus), 'docs |', len(qs_all), 'questions')
print(collections.Counter((q['track'], q['subtype']) for q in qs_all))

## 4. Choose models
Use **one model per row**; all systems in a run share it. Groq retires models over time: the cell prints what your key can see.
Recommended: one Llama (e.g. `llama-3.3-70b-versatile` if available) + `openai/gpt-oss-120b` (+ optionally a Qwen model).

In [ ]:
from groq import Groq
avail = sorted(m.id for m in Groq().models.list().data)
print('\n'.join(avail))
MODELS  = ['llama-3.3-70b-versatile', 'openai/gpt-oss-120b']   # edit to what is available
REPEATS = [0, 1, 2]        # 3 repeats per model for variance (start with [0])
MODELS = [m for m in MODELS if m in avail]; MODELS

## 5. Smoke test on one real-conflict question

In [ ]:
from ctirag.systems import SYSTEMS
llm = make_llm(make_cfg(llm_model=MODELS[0]), os.environ['GROQ_API_KEY'])
q = next(q for q in qs_all if q['subtype'] == 'version_drift' and q['meta']['drift'] == 'removed')
print(q['question'], '\nGOLD:', q['gold_ids'], ' OUTDATED (v8.2 only):', q['stale_ids'])
for name in ['graph_oracle', 'naive', 'evidence_agent']:
    r = SYSTEMS[name](corpus, index, llm, cfg).run(q['question'])
    print(f"\n[{name}] ids={r['ids']} flags={r['flags']} calls={r['usage'].get('calls')}\n{r['answer'][:300]}")

## 6. Pilot (3 questions per family) + budget estimate

In [ ]:
pilot = stratified_sample(qs_all, 3, cfg.seed)
pcfg = make_cfg(llm_model=MODELS[0], repeat=0)
plm = make_llm(pcfg, os.environ['GROQ_API_KEY'])
for s in pcfg.systems:
    run_system(s, corpus, index, plm, pcfg, pilot, run_tag=run_tag(plm.model, 0, 'pilot'), workers=3)
dfp = score_run(pilot, load_results(pcfg.run_dir / run_tag(plm.model, 0, 'pilot'), pcfg.systems), corpus)
per_q = dfp.groupby('system')[['calls', 'prompt_tokens', 'completion_tokens']].mean()
full = per_q * len(qs_all)
print('Per question:'); display(per_q.round(1))
print(f'One full run ({len(qs_all)} questions, one model, one repeat):')
print(' calls  :', int(full.calls.sum()))
print(' tokens :', int(full.prompt_tokens.sum() + full.completion_tokens.sum()))
print(f'x {len(MODELS)} models x {len(REPEATS)} repeats -> multiply accordingly')

## 7. Full runs (resumable)
Each (model, repeat) writes to its own folder. If Groq's daily quota runs out the loop stops cleanly; re-run the cell later and it continues.
Set `PER_SUBTYPE` to e.g. 25 to cut cost (≈ 400 questions) while keeping every family.

In [ ]:
PER_SUBTYPE = None
qs = stratified_sample(qs_all, PER_SUBTYPE, cfg.seed)
for model in MODELS:
    for rep in REPEATS:
        rcfg = make_cfg(llm_model=model, repeat=rep, temperature=0.0 if rep == 0 else 0.3)
        rllm = make_llm(rcfg, os.environ['GROQ_API_KEY'])
        tag = run_tag(rllm.model, rep)
        (rcfg.run_dir / tag).mkdir(parents=True, exist_ok=True)
        (rcfg.run_dir / tag / 'meta.json').write_text(json.dumps({'model': rllm.model, 'repeat': rep,
            'qids': [q['qid'] for q in qs], **rcfg.to_dict()}, indent=2))
        for s in rcfg.systems:
            run_system(s, corpus, index, rllm, rcfg, qs, run_tag=tag, workers=4)

## 8. Reports: one per run, then the multi-model table

In [ ]:
from IPython.display import Markdown, Image, display
dfs = {}
for model in MODELS:
    for rep in REPEATS:
        tag = run_tag(model, rep)
        d = cfg.run_dir / tag
        if not (d / 'meta.json').exists():
            continue
        df = score_run(qs, load_results(d, cfg.systems), corpus)
        dfs[f'{model}/r{rep}'] = df
        if rep == 0:
            make_report(df, d / 'report', model)
            for t in ['table_main_text', 'table_main_evidence', 'table_temporal', 'table_claims', 'table_significance']:
                p = d / 'report' / f'{t}.md'
                if p.exists(): display(Markdown(p.read_text()))
            for f in ['fig_main_metrics', 'fig_f1_heatmap', 'fig_uar_breakdown', 'fig_cost_quality']:
                display(Image(str(d / 'report' / f'{f}.png')))
display(Markdown(multi_model_table(dfs, cfg.run_dir / 'main').to_markdown(index=False)))

## 9. Human validation of the generated gold (required for the paper)
1. Export the sheet, give one copy to you and one to a colleague. Label each row CORRECT / INCORRECT / AMBIGUOUS **independently**.
2. Upload both filled sheets and compute Cohen's kappa. Report kappa and gold accuracy; remove items both marked INCORRECT and re-score.

In [ ]:
from ctirag.annotate import export_sample, agreement, drop_rejected
from pathlib import Path
export_sample(qs_all, corpus, Path(WORK) / 'annotation_sheet.csv', n=150)

In [ ]:
# after both annotators are done:
# res = agreement(Path(WORK)/'annotation_A.csv', Path(WORK)/'annotation_B.csv', Path(WORK)/'annotation')
# qs_clean = drop_rejected(qs, Path(WORK)/'annotation_A.csv', Path(WORK)/'annotation_B.csv')

## 10. Error analysis: plausible-but-unsupported associations and stale adoption

In [ ]:
model = MODELS[0]; d = cfg.run_dir / run_tag(model, 0)
res = load_results(d, cfg.systems); df = dfs[f'{model}/r0']
qmap = {q['qid']: q for q in qs}
bad = df[(df.system == 'naive') & ((df.uar_plausible > 0) | (df.stale_adoption == 1))].qid.tolist()[:10]
for qid in bad:
    q = qmap[qid]
    print('=' * 100); print(q['track'], q['subtype'], '|', q['question'][:250])
    print('GOLD :', q['gold_ids'] or q['gold_label'], '| stale/negative:', q['stale_ids'])
    for s in ['naive', 'react_agent', 'evidence_agent']:
        r = res[s][qid]
        print(f'{s:15s} ids={r["ids"]} verdict={r["verdict"]} flags={r["flags"]}', ('rejected=' + str(r.get('rejected'))[:200]) if 'rejected' in r else '')

## 11. (Optional) Injected-conflict stress test
Low-authority notes from four source types with five wording templates. Report it as a *stress test*, separately from the real conflicts above.

In [ ]:
import json as _j
from ctirag.benchmark import BenchmarkBuilder
from ctirag.corpus import Corpus
scfg = make_cfg(inject_conflicts=True, n_poisoned=40, llm_model=MODELS[0])
base = load_corpus(scfg, with_notes=False)
procs = [_j.loads(l) for l in open(scfg.proc_dir / 'procedures.jsonl')]
b = BenchmarkBuilder(base, procs, scfg); pq = b.poisoned(40)
for n in b.notes: base.add(n)
six = HybridIndex(base, scfg.dense_model, scfg.proc_dir)
sllm = make_llm(scfg, os.environ['GROQ_API_KEY'])
for s in ['naive', 'kg_rag', 'crag', 'react_agent', 'evidence_agent', 'agent_no_authority']:
    run_system(s, base, six, sllm, scfg, pq, run_tag=run_tag(sllm.model, 0, 'stress'), workers=4)
dst = score_run(pq, load_results(scfg.run_dir / run_tag(sllm.model, 0, 'stress'), scfg.systems), base)
display(dst.groupby('system')[['f1', 'stale_adoption', 'flag_correct']].mean().mul(100).round(1))